# Phase 3 — Survival / retention

Licence stay time from the sponsor panel. Exits are right-censored at the latest snapshot when still active.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from lifelines import KaplanMeierFitter

cwd = Path.cwd()
project_root = cwd if (cwd / "data" / "processed").exists() else cwd.parent
sys.path.insert(0, str(project_root / "src"))

from run_survival import run_survival

results = run_survival()
surv = pd.read_parquet(project_root / "data" / "processed" / "survival_table.parquet")
scores = pd.read_parquet(project_root / "data" / "processed" / "sponsor_retention_scores.parquet")
print(surv[["duration_days", "event", "rating", "region"]].head())
results["n_companies"], results["n_events"]

In [ ]:
km = KaplanMeierFitter()
ax = plt.subplot(111)
for rating, chunk in surv.groupby("rating"):
    km.fit(chunk["duration_days"].clip(lower=1), chunk["event"], label=str(rating))
    km.plot_survival_function(ax=ax)
plt.title("KM by rating")
plt.xlabel("Days")
plt.ylabel("Survival")
plt.tight_layout()
plt.show()

In [ ]:
print("Log-rank rating p:", results["logrank_rating_p"])
print("Log-rank region p:", results["logrank_region_p"])
print("PH check:", results["ph_check"])
display(results["cox_summary"][["coef", "exp(coef)", "p", "coef lower 95%", "coef upper 95%"]])
scores.sort_values("retention_risk_score", ascending=False).head(10)

## Limitations

- Snapshot gaps mean exit timing is approximate (interval censoring simplified to event at last_seen).
- No sector covariate in the Home Office register.
- Identity is cleaned company name only.